# CVAT → YOLOv11 Converter
**Pickleball Ball Detection Dataset Preparation**

Run each cell in order. The output dataset will be saved to your Google Drive.

In [ ]:
# ── CELL 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── CELL 2: Configuration ────────────────────────────────────────────────
# List all your clip folders here
CLIP_FOLDERS = [
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_1',
    '/content/drive/MyDrive/In_Out_Pickleball/game_1/clip_2',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_1',
    '/content/drive/MyDrive/In_Out_Pickleball/game_2/clip_2',
    '/content/drive/MyDrive/In_Out_Pickleball/game_5/clip_1',
]

# Output dataset folder (will be created on your Drive)
OUTPUT_DIR = '/content/drive/MyDrive/In_Out_Pickleball/yolo_dataset'

# Image resolution of your frames
IMG_WIDTH  = 1920
IMG_HEIGHT = 1080

# Bounding box size around ball center point (pixels)
# Increase if ball looks large in your footage, decrease if tiny
BALL_BOX_PX = 40

# Train / val split
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

# ── What is the XML file called inside each clip folder? ──
# CVAT usually exports as 'annotations.xml'
# Run Cell 3 first to auto-detect the actual name
XML_FILENAME = 'annotations.xml'

print('Configuration set ✓')

In [ ]:
# ── CELL 3: Inspect folders — run this FIRST to check your files ────────
import os
from pathlib import Path

print('=' * 60)
print('FOLDER INSPECTION REPORT')
print('=' * 60)

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

for clip_path in CLIP_FOLDERS:
    p = Path(clip_path)
    print(f'\n📁 {clip_path}')

    if not p.exists():
        print('   ❌ FOLDER NOT FOUND — check the path above')
        continue

    all_files = list(p.iterdir())
    xml_files = [f for f in all_files if f.suffix.lower() == '.xml']
    img_files = [f for f in all_files if f.suffix.lower() in IMG_EXTENSIONS]
    csv_files = [f for f in all_files if f.suffix.lower() == '.csv']

    # Check for frames subfolder
    frames_sub = p / 'frames'
    if frames_sub.exists():
        sub_imgs = [f for f in frames_sub.iterdir() if f.suffix.lower() in IMG_EXTENSIONS]
        sub_xmls = [f for f in frames_sub.iterdir() if f.suffix.lower() == '.xml']
        print(f'   📂 frames/ subfolder found:')
        print(f'      Images : {len(sub_imgs)}')
        print(f'      XML    : {[x.name for x in sub_xmls]}')
        if sub_xmls:
            print(f'   ⚠️  NOTE: Update XML_FILENAME and CLIP_FOLDERS to point inside frames/')
    else:
        print(f'   Images : {len(img_files)}')
        print(f'   XML    : {[x.name for x in xml_files]}')
        print(f'   CSV    : {[x.name for x in csv_files]}')

        if img_files:
            sample = sorted(img_files)[:3]
            print(f'   Sample : {[f.name for f in sample]}')

print('\n' + '=' * 60)
print('If XML files have a different name, update XML_FILENAME in Cell 2')

In [ ]:
# ── CELL 4: Parse one XML as a sample — verify annotations look correct ──
import xml.etree.ElementTree as ET

def parse_cvat_xml(xml_path, img_w, img_h):
    """
    Parse CVAT Images 1.1 XML.
    Returns dict: { image_name: {x, y, visible} }
    Handles both <points> (point tool) and <box> elements.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = {}

    for image_elem in root.findall('image'):
        img_name = image_elem.get('name', '')
        img_name = Path(img_name).name  # strip any path prefix CVAT adds

        # ── Check <points> element (point annotation tool) ──
        for pts_elem in image_elem.findall('points'):
            occluded = pts_elem.get('occluded', '0')
            visible  = (occluded == '0')

            # Check custom 'visibility' attribute tag
            for attr in pts_elem.findall('attribute'):
                if attr.get('name', '').lower() == 'visibility':
                    val = (attr.text or '1').strip()
                    visible = (val == '1')

            pts_str = pts_elem.get('points', '')
            if not pts_str:
                continue
            try:
                x_str, y_str = pts_str.split(';')[0].split(',')
                x, y = float(x_str), float(y_str)
            except ValueError:
                continue

            # Out of bounds = invisible
            if not (0 <= x <= img_w and 0 <= y <= img_h):
                visible = False

            annotations[img_name] = {'x': x, 'y': y, 'visible': visible}
            break

        # ── Fallback: <box> element ──
        if img_name not in annotations:
            for box_elem in image_elem.findall('box'):
                occluded = box_elem.get('occluded', '0')
                visible  = (occluded == '0')
                xtl = float(box_elem.get('xtl', 0))
                ytl = float(box_elem.get('ytl', 0))
                xbr = float(box_elem.get('xbr', 0))
                ybr = float(box_elem.get('ybr', 0))
                annotations[img_name] = {
                    'x': (xtl + xbr) / 2,
                    'y': (ytl + ybr) / 2,
                    'visible': visible
                }
                break

    return annotations


# Test on first clip folder
test_clip = Path(CLIP_FOLDERS[0])
test_xml  = test_clip / XML_FILENAME

if not test_xml.exists():
    # Try inside frames/ subfolder
    test_xml = test_clip / 'frames' / XML_FILENAME

if test_xml.exists():
    ann = parse_cvat_xml(test_xml, IMG_WIDTH, IMG_HEIGHT)
    visible   = {k: v for k, v in ann.items() if v['visible']}
    invisible = {k: v for k, v in ann.items() if not v['visible']}

    print(f'XML parsed: {test_xml}')
    print(f'Total annotated frames : {len(ann)}')
    print(f'  Visible (ball shown) : {len(visible)}')
    print(f'  Invisible            : {len(invisible)}')
    print()
    print('Sample visible annotations (first 5):')
    for name, data in list(visible.items())[:5]:
        print(f'  {name:40s}  x={data["x"]:8.2f}  y={data["y"]:8.2f}')
    print()
    if invisible:
        print('Sample invisible annotations (first 3):')
        for name, data in list(invisible.items())[:3]:
            print(f'  {name}')
else:
    print(f'❌ XML not found at: {test_xml}')
    print('   Update XML_FILENAME in Cell 2 or check your folder structure with Cell 3')

In [ ]:
# ── CELL 5: Visualize sample annotations — confirm box size is correct ───
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

random.seed(RANDOM_SEED)

test_clip = Path(CLIP_FOLDERS[0])
test_xml  = test_clip / XML_FILENAME
if not test_xml.exists():
    test_xml = test_clip / 'frames' / XML_FILENAME
    test_clip = test_clip / 'frames'

ann = parse_cvat_xml(test_xml, IMG_WIDTH, IMG_HEIGHT)
visible_frames = [(k, v) for k, v in ann.items() if v['visible']]

if not visible_frames:
    print('No visible frames found in first clip')
else:
    # Pick up to 4 random visible frames
    samples = random.sample(visible_frames, min(4, len(visible_frames)))
    fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, (img_name, data) in zip(axes, samples):
        img_path = test_clip / img_name
        if not img_path.exists():
            ax.set_title(f'NOT FOUND:\n{img_name}', fontsize=8)
            continue

        img = Image.open(img_path)
        ax.imshow(img)

        # Draw bounding box
        half = BALL_BOX_PX / 2
        rect = patches.Rectangle(
            (data['x'] - half, data['y'] - half),
            BALL_BOX_PX, BALL_BOX_PX,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
        ax.plot(data['x'], data['y'], 'r+', markersize=12, markeredgewidth=2)
        ax.set_title(f'{img_name}\nx={data["x"]:.0f} y={data["y"]:.0f}', fontsize=8)
        ax.axis('off')

    plt.suptitle(f'Sample annotations — box size: {BALL_BOX_PX}px\n'
                 f'Green box = YOLO label | Red + = center point',
                 fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'\nIf the green box looks too big/small, adjust BALL_BOX_PX in Cell 2 and re-run.')

In [ ]:
# ── CELL 6: Convert ALL clips and build YOLO dataset ────────────────────
import shutil
import random as rnd

rnd.seed(RANDOM_SEED)
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

def to_yolo_label(x_px, y_px, box_px, img_w, img_h):
    half = box_px / 2
    x1 = max(0.0, x_px - half)
    y1 = max(0.0, y_px - half)
    x2 = min(float(img_w), x_px + half)
    y2 = min(float(img_h), y_px + half)
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'

# Create output dirs
out = Path(OUTPUT_DIR)
for split in ['train', 'val']:
    (out / 'images' / split).mkdir(parents=True, exist_ok=True)
    (out / 'labels' / split).mkdir(parents=True, exist_ok=True)

all_entries = []  # (img_path, label_str_or_None, clip_tag)
summary = []

for clip_path in CLIP_FOLDERS:
    clip_dir = Path(clip_path)
    xml_path = clip_dir / XML_FILENAME

    # Handle frames/ subfolder layout
    if not xml_path.exists() and (clip_dir / 'frames' / XML_FILENAME).exists():
        clip_dir = clip_dir / 'frames'
        xml_path = clip_dir / XML_FILENAME

    if not xml_path.exists():
        print(f'⚠️  Skipping {clip_path} — {XML_FILENAME} not found')
        continue

    ann = parse_cvat_xml(xml_path, IMG_WIDTH, IMG_HEIGHT)
    images = sorted([f for f in clip_dir.iterdir()
                     if f.suffix.lower() in IMG_EXTENSIONS])

    # Build a unique prefix from the clip path to avoid filename collisions
    # e.g. game_1__clip_1
    parts = Path(clip_path).parts
    tag   = '__'.join(parts[-2:])  # e.g. game_1__clip_1

    labeled = invisible = no_ann = 0
    for img in images:
        data = ann.get(img.name)
        if data is None:
            all_entries.append((img, None, tag))
            no_ann += 1
        elif not data['visible']:
            all_entries.append((img, None, tag))
            invisible += 1
        else:
            lbl = to_yolo_label(data['x'], data['y'], BALL_BOX_PX, IMG_WIDTH, IMG_HEIGHT)
            all_entries.append((img, lbl, tag))
            labeled += 1

    summary.append((tag, len(images), labeled, invisible, no_ann))

# ── Print summary ──
print(f'{"Clip":<25} {"Total":>7} {"Ball":>7} {"Invisible":>10} {"No ann":>8}')
print('-' * 60)
for tag, total, labeled, invisible, no_ann in summary:
    print(f'{tag:<25} {total:>7} {labeled:>7} {invisible:>10} {no_ann:>8}')
print('-' * 60)
totals = [sum(x[i] for x in summary) for i in range(1, 5)]
print(f'{"TOTAL":<25} {totals[0]:>7} {totals[1]:>7} {totals[2]:>10} {totals[3]:>8}')

# ── Train / val split ──
rnd.shuffle(all_entries)
split_idx = int(len(all_entries) * TRAIN_RATIO)
train_entries = all_entries[:split_idx]
val_entries   = all_entries[split_idx:]

print(f'\nSplit → Train: {len(train_entries)} | Val: {len(val_entries)}')
print('\nCopying files... (this may take a few minutes for 2000+ images)')

def write_split(entries, split_name):
    img_dir = out / 'images' / split_name
    lbl_dir = out / 'labels' / split_name
    for img_path, label_str, tag in entries:
        safe_name = f'{tag}__{img_path.name}'
        shutil.copy2(img_path, img_dir / safe_name)
        if label_str is not None:
            (lbl_dir / (Path(safe_name).stem + '.txt')).write_text(label_str)
    print(f'  {split_name}: {len(entries)} images written ✓')

write_split(train_entries, 'train')
write_split(val_entries,   'val')
print('\nAll files copied ✓')

In [ ]:
# ── CELL 7: Write data.yaml ───────────────────────────────────────────────
yaml_content = f"""# YOLOv11 Dataset — Pickleball Ball Detection
# Auto-generated

path: {OUTPUT_DIR}
train: images/train
val:   images/val

nc: 1
names:
  0: ball
"""
yaml_path = out / 'data.yaml'
yaml_path.write_text(yaml_content)
print(f'data.yaml written ✓')
print(f'Path: {yaml_path}')
print()
print(yaml_content)

In [ ]:
# ── CELL 8: Final verification — check output structure ──────────────────
out = Path(OUTPUT_DIR)

train_imgs = list((out / 'images' / 'train').iterdir())
val_imgs   = list((out / 'images' / 'val').iterdir())
train_lbls = list((out / 'labels' / 'train').iterdir())
val_lbls   = list((out / 'labels' / 'val').iterdir())

print('=' * 50)
print('OUTPUT DATASET SUMMARY')
print('=' * 50)
print(f'images/train : {len(train_imgs):>5} files')
print(f'images/val   : {len(val_imgs):>5} files')
print(f'labels/train : {len(train_lbls):>5} .txt files  (frames with visible ball)')
print(f'labels/val   : {len(val_lbls):>5} .txt files')
print()
print(f'Label rate train : {len(train_lbls)/len(train_imgs)*100:.1f}%')
print(f'Label rate val   : {len(val_lbls)/len(val_imgs)*100:.1f}%')

# Show a sample label file
if train_lbls:
    sample_lbl = sorted(train_lbls)[0]
    print(f'\nSample label file: {sample_lbl.name}')
    print(f'Content: {sample_lbl.read_text().strip()}')
    print('Format:  class  cx      cy      w       h')

print()
print('Dataset is ready for YOLOv11 training!')
print(f'data.yaml: {OUTPUT_DIR}/data.yaml')

In [ ]:
# ── CELL 9 (OPTIONAL): Train YOLOv11 ─────────────────────────────────────
# Only run this after verifying the dataset looks correct in Cell 5 & 8

# !pip install ultralytics -q

# from ultralytics import YOLO

# model = YOLO('yolo11n.pt')   # nano = fastest, good for testing
# # model = YOLO('yolo11s.pt') # small = better accuracy

# results = model.train(
#     data  = f'{OUTPUT_DIR}/data.yaml',
#     epochs = 100,
#     imgsz  = 640,
#     batch  = 16,
#     name   = 'pickleball_ball_v1',
#     project= '/content/drive/MyDrive/In_Out_Pickleball/runs',
# )
print('Uncomment the code above when ready to train.')